# Composite well log panels — Slochteren formation


Per-well log displays (GR, RHOB, porosity, Vshale) over the Slochteren TVD interval.

## Outputs
- `05_visualizations/well_logs/blt01_log_panel_v01.png` — detailed BLT-01 panel
- `05_visualizations/well_logs/four_wells_comparison_v01.png` — side-by-side comparison


In [ ]:
import sys
sys.path.insert(0, '../../../06_ai_workflow_bonus/pipeline')
from pipeline import run_pipeline

# Load all four wells via the AI workflow pipeline
wells_meta = [('BLT-01','blt01'), ('EVD-01','evd01'), ('JUT-01','jut01'), ('PKP-01','pkp01')]
wells_data = {}
for w, slug in wells_meta:
    df, summary, _ = run_pipeline(
        well_id=w,
        las_path=f'../../../01_data_raw/well_logs_las/{slug}.las',
        survey_path=f'../../../02_data_processed/tvd_conversions/{slug}_deviation_survey.csv',
        litho_path='../../../01_data_raw/lithostratigraphy/lithostratigraphic_data.xlsx',
        litho_sheet=w
    )
    wells_data[w] = (df, summary)
    print(f"{w}: top {summary['top_tvd_m']}, base {summary['base_tvd_m']}, ϕ_eff {summary['phie_mean']*100:.1f}%")

## Plotting logic

See `03_analysis/scripts/plotting_utils.py` for the `composite_log_panel` and `four_wells_panel` helpers used to produce the figures.

In [ ]:
# --- Detailed BLT-01 composite log panel ---
import matplotlib.pyplot as plt
sys.path.insert(0, '../../../03_analysis/scripts')
from plotting_utils import composite_log_panel, apply_style, COLORS
apply_style()

df_blt, sum_blt = wells_data['BLT-01']
fig, axes = composite_log_panel(df_blt, sum_blt['top_tvd_m'], sum_blt['base_tvd_m'],
                                title='BLT-01 \u2014 Slochteren composite log')
fig.savefig('../../../05_visualizations/well_logs/blt01_log_panel_v01.png', dpi=150, bbox_inches='tight')
fig.savefig('../../../05_visualizations/well_logs/blt01_log_panel_v01.svg', bbox_inches='tight')
plt.show()

In [ ]:
# --- Four-well side-by-side comparison (GR + effective porosity) ---
fig, axes = plt.subplots(1, 4, figsize=(14, 8))
for ax, (w, _slug) in zip(axes, wells_meta):
    df, s = wells_data[w]
    top, base = s['top_tvd_m'], s['base_tvd_m']
    win = df[(df['TVD'] >= top - 30) & (df['TVD'] <= base + 30)]
    ax.plot(win['GR'], win['TVD'], color=COLORS['shale'], lw=0.6)
    if 'PHIE' in win.columns:
        ax2 = ax.twiny()
        ax2.fill_betweenx(win['TVD'], 0, win['PHIE'].clip(lower=0) * 100,
                          color=COLORS['sand'], alpha=0.5)
        ax2.set_xlim(0, 40); ax2.set_xticks([0, 20, 40]); ax2.tick_params(labelsize=7)
        ax2.set_xlabel('\u03d5e (%)', fontsize=7)
    ax.axhspan(top, base, color=COLORS['sand'], alpha=0.15)
    ax.axvline(60, color='gray', ls='--', alpha=0.5)
    ax.set_xlim(0, 200); ax.invert_yaxis()
    ax.set_title(f"{w}\n\u03d5e {s['phie_mean']*100:.1f}%  NTG {s['ntg']:.2f}", fontsize=9, fontweight='bold')
    ax.set_xlabel('GR (gAPI)')
axes[0].set_ylabel('TVD (m)')
plt.suptitle('Four-well Slochteren comparison (GR curve + effective-porosity fill)', fontweight='bold')
plt.tight_layout()
fig.savefig('../../../05_visualizations/well_logs/four_wells_comparison_v01.png', dpi=150, bbox_inches='tight')
fig.savefig('../../../05_visualizations/well_logs/four_wells_comparison_v01.svg', bbox_inches='tight')
plt.show()